# Kogod AI Survey Analysis (2024–2026)

## Revised portfolio edition · ITEC-400 / 600 team project

**Assem Mendygaziyeva** · Course project developed with Natalie Cope, Victoria Smith, and Esha Teware.

<hr>

## Project Overview:

This project describes AI usage, attitudes, and reported behaviors across three years of Kogod survey responses. Year comparisons describe the sampled responses; participant linkage is unavailable, so changes are not interpreted as within-person change or causal effects.

This revised notebook consumes a previously cleaned combined survey file. It produces descriptive comparisons and visualizations. Optional qualitative charts consume precomputed LLM annotations; annotation generation and validation are not included. The original notebook imported statistical-test functions but did not run them, so no inferential testing is claimed.

## Research Question:

The primary research question guiding this analysis is:
How has AI adoption changed among Kogod students from 2024 to 2026, and how do Power Users differ from Normal Users in confidence, behavior, and academic risk patterns?

## Import libraries:

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
sns.set_theme(style="whitegrid")
DATA_DIR = Path(os.environ.get("KOGOD_DATA_DIR", "data")).expanduser()
YEARS = [2024, 2025, 2026]
USAGE_ORDER = ["0", "1-3", "4-7", "8-10", "11+"]
GROUP_ORDER = ["Non-user", "Normal User", "Power User"]

def flag(series):
    values = series.astype("string").str.strip().str.lower()
    invalid = values.notna() & ~values.isin(["true", "false", "1", "0"])
    if invalid.any():
        raise ValueError("Unexpected Boolean values in cohort indicator")
    return values.isin(["true", "1"])

def classify_usage(value):
    if pd.isna(value):
        return "Unknown"
    bucket = str(value).strip()
    return {"0": "Non-user", "1-3": "Normal User", "4-7": "Normal User",
            "8-10": "Power User", "11+": "Power User"}.get(bucket, "Unknown")

def prepare(frame):
    frame = frame.copy()
    frame["year"] = pd.to_numeric(frame["year"], errors="raise").astype(int)
    if not frame["year"].isin(YEARS).all():
        raise ValueError("Unexpected survey year")
    frame["usage_bucket"] = frame["usage_bucket"].astype("string").str.strip()
    invalid = frame["usage_bucket"].notna() & ~frame["usage_bucket"].isin(USAGE_ORDER)
    if invalid.any():
        raise ValueError("Unexpected usage bucket; review source coding")
    frame["user_group"] = frame["usage_bucket"].map(classify_usage)
    return frame

def load_kogod():
    return df_kogod.copy()

## Load 2024 \- 2026 files:

In [ ]:
input_path = DATA_DIR / "cleaned_master_all_years.csv"
if not input_path.exists():
    raise FileNotFoundError("Supply the authorized cleaned_master_all_years.csv via KOGOD_DATA_DIR; survey data are not distributed.")
df_all = pd.read_csv(input_path)
required = {"year", "usage_bucket", "is_kogod_major", "ai_study_skill", "ai_prepared_career",
            "ai_clear_policy", "ai_improved_study", "ai_efficiency", "ai_shortcut",
            "model_favor", "model_factor", "perplexity_usage"}
missing = required.difference(df_all.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")
df_all = prepare(df_all)
df_kogod = df_all.loc[flag(df_all["is_kogod_major"])].copy()
if df_kogod.empty:
    raise ValueError("No Kogod respondents identified")

Checking data structure:

In [ ]:
# Aggregate checks only; never display respondent rows or open-ended responses.
print("All survey records:", len(df_all))
print("Kogod survey records:", len(df_kogod))
print(df_kogod.groupby("year").size().rename("Kogod records"))

## Consistent user groups

Power Users report 8+ uses per week; Normal Users report 1–7; Non-users report zero. Missing usage is Unknown and excluded only from usage-based comparisons. Groups are defined from the usage bucket throughout, rather than mixing definitions or treating missing flags as non-use.

In [ ]:
print(df_kogod["user_group"].value_counts().reindex(GROUP_ORDER + ["Unknown"], fill_value=0))

In [ ]:
df_trend = df_kogod.dropna(subset=["usage_bucket"])

usage_trend = (
    df_trend
    .groupby(["year", "usage_bucket"])
    .size()
    .reset_index(name="count")
)

usage_trend["percent"] = usage_trend.groupby("year")["count"].transform(lambda x: x / x.sum())

## Overall Usage Trends \(2024–2026\) Plots:

### Weekly Usage Frequency by Year \(Kogod\-only\) \- Bar Chart:

In [ ]:
order = ["0", "1-3", "4-7", "8-10", "11+"]

usage_trend["usage_bucket"] = pd.Categorical(
    usage_trend["usage_bucket"],
    categories=order,
    ordered=True
)

plt.figure(figsize=(8,5))

ax = sns.barplot(
    data=usage_trend,
    x="usage_bucket",
    y="percent",
    hue="year"
)

plt.title("Weekly AI Usage Frequency by Year (Kogod-only)", fontsize=11)
plt.xlabel("Times per week")
plt.ylabel("% of respondents")

plt.gca().yaxis.set_major_formatter(lambda x, _: f'{int(x*100)}%')

for container in ax.containers:
    ax.bar_label(container, fmt=lambda x: f'{int(x*100)}%', fontsize=8)

plt.legend(title="year", fontsize=8, title_fontsize=9)
sns.despine()

plt.tight_layout()
plt.show()

### Power User Share Shift \(Kogod\-Only\) \- Line Plot

In [ ]:
df_trend = df_kogod.dropna(subset=["usage_bucket"]).copy()

df_trend["is_power_user_calc"] = df_trend["usage_bucket"].isin(["8-10", "11+"]).astype(int)

power_trend = (
    df_trend
    .groupby("year")["is_power_user_calc"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(6,4))

ax = sns.lineplot(
    data=power_trend,
    x="year",
    y="is_power_user_calc",
    marker="o",
    color="#4C72B0"
)

plt.xticks([2024, 2025, 2026])

for i, row in power_trend.iterrows():
    ax.text(row["year"], row["is_power_user_calc"], f"{round(row['is_power_user_calc']*100,1)}%", 
            ha='center', va='bottom', fontsize=9)

plt.title("Power User Share Shift, 2024 → 2025 → 2026 (Kogod-only)", fontsize=11)
plt.xlabel("Year")
plt.ylabel("Power Users (% of valid usage responses)")

plt.gca().yaxis.set_major_formatter(lambda x, _: f'{int(x*100)}%')

sns.despine()

plt.tight_layout()
plt.show()

### Agreement-response distributions by year

The source scoring merges some response categories. Without the survey codebook, no revised numeric scale is assumed. These charts retain the observed response labels; percentages use nonmissing responses per item and year. This avoids treating ordinal intervals as validated numeric distances.

In [ ]:
LIKERT_COLS = {
    "ai_study_skill": "AI skills for academic purposes",
    "ai_prepared_career": "Prepared to use AI for job/internship",
    "ai_clear_policy": "Instructor AI policy clarity",
    "ai_improved_study": "AI improved understanding",
    "ai_efficiency": "AI increased efficiency"
}
RESPONSE_ORDER = ["strongly disagree", "disagree", "somewhat disagree",
                  "neither agree nor disagree", "somewhat agree", "agree", "strongly agree"]
likert_tables = {}
for col in LIKERT_COLS:
    responses = df_kogod[col].astype("string").str.strip().str.lower()
    responses = responses.replace({"neither disagree nor agree": "neither agree nor disagree",
                                   "neutral": "neither agree nor disagree"})
    invalid = responses.notna() & ~responses.isin(RESPONSE_ORDER)
    if invalid.any():
        raise ValueError(f"Unexpected response category for {col}")
    tab = pd.crosstab(df_kogod["year"], responses).reindex(index=YEARS, columns=RESPONSE_ORDER, fill_value=0)
    denominator = tab.sum(axis=1).replace(0, np.nan)
    likert_tables[col] = tab.div(denominator, axis=0) * 100
    print(LIKERT_COLS[col], "valid response counts:", tab.sum(axis=1).to_dict())

In [ ]:
fig, axes = plt.subplots(len(LIKERT_COLS), 1, figsize=(11, 18))
for ax, (col, label) in zip(axes, LIKERT_COLS.items()):
    likert_tables[col].plot.bar(stacked=True, ax=ax, colormap="RdYlBu", legend=False)
    ax.set_title(label)
    ax.set_ylabel("Valid responses (%)")
    ax.set_ylim(0, 100)
    ax.set_xlabel("Survey year")
    ax.tick_params(axis="x", rotation=0)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3)
fig.tight_layout(rect=(0, 0.06, 1, 1))
plt.show()

## Optional analysis of precomputed LLM annotations

This section runs only when an authorized `AI_Survey_LLM_Analysis.xlsx` is available. No annotations are fabricated. Missing labels are excluded from each theme’s denominator, rather than treated as absence. Percentages can exceed 100% across themes because responses may have multiple tags. LLM model, prompts, coding reliability, and validation are unavailable, so results are exploratory.

In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
annotation_path = DATA_DIR / "AI_Survey_LLM_Analysis.xlsx"
annotations_available = annotation_path.exists()
if annotations_available:
    df = pd.read_excel(annotation_path, sheet_name="Full_Data")
    if "is_kogod_major" not in df:
        raise ValueError("Annotation file needs is_kogod_major to verify the Kogod cohort")
    df = prepare(df.loc[flag(df["is_kogod_major"])])
    theme_cols = [c for c in df if c.startswith("concern_") or c.startswith("skill_")]
    for col in theme_cols:
        df[col] = pd.to_numeric(df[col], errors="raise")
        if not df[col].dropna().isin([0, 1]).all():
            raise ValueError("Theme labels must be 0, 1, or missing")
    if "perplexity_sentiment" in df:
        df["perplexity_sentiment"] = df["perplexity_sentiment"].astype("string").str.strip().str.lower()
        if not df["perplexity_sentiment"].dropna().isin(["positive", "neutral", "negative"]).all():
            raise ValueError("Unexpected sentiment label")
else:
    print("Optional LLM annotation file unavailable; qualitative analysis skipped.")

In [ ]:
if annotations_available:
    concern_themes = {"concern_over_reliance_skill_loss": "Over-reliance/Skill Loss",
        "concern_accuracy_misinformation": "Accuracy/Misinformation", "concern_job_displacement": "Job Displacement",
        "concern_academic_integrity": "Academic Integrity", "concern_privacy_security": "Privacy/Security",
        "concern_environmental": "Environmental", "concern_ethical_bias": "Ethical/Bias"}
    skill_themes = {"skill_prompting": "Prompting", "skill_ethical_responsible_use": "Ethical use",
        "skill_subject_specific": "Subject-specific applications", "skill_learning_not_substitution": "Learning",
        "skill_tool_selection": "Tool selection", "skill_verification_accuracy": "Verification"}
    def theme_summary(frame, group, response_col, themes):
        required = {response_col, *themes}
        if required.difference(frame.columns):
            raise ValueError("Annotation schema is incomplete")
        frame = frame.loc[frame[response_col].notna()]
        rows = []
        for label, part in frame.groupby(group):
            if label == "Unknown":
                continue
            for col, theme in themes.items():
                valid = part[col].dropna()
                rows.append({group: label, "theme": theme, "labeled_n": len(valid),
                             "percent": valid.mean() * 100 if len(valid) else np.nan})
        return pd.DataFrame(rows)
    for response_col, themes, title in [("ai_concern_open", concern_themes, "AI concerns"),
                                      ("desire_ai_skill", skill_themes, "Desired skills")]:
        for grouping in ["year", "user_group"]:
            summary = theme_summary(df, grouping, response_col, themes)
            print(summary.to_string(index=False))
            if not summary.empty:
                go.Figure([go.Bar(name=str(label), x=part["theme"], y=part["percent"])
                    for label, part in summary.groupby(grouping)]).update_layout(
                        title=f"{title} by {grouping}", barmode="group", yaxis_title="Labeled responses (%)").show()
    if not {"perplexity_feedback", "perplexity_sentiment"}.issubset(df.columns):
        raise ValueError("Missing feedback/sentiment columns")
    feedback = df.loc[df["year"].isin([2025, 2026]) & df["perplexity_feedback"].notna()]
    for grouping in ["year", "user_group"]:
        valid = feedback.loc[feedback["perplexity_sentiment"].notna()]
        if grouping == "user_group":
            valid = valid.loc[valid["user_group"].ne("Unknown")]
        table = pd.crosstab(valid[grouping], valid["perplexity_sentiment"])
        pct = table.div(table.sum(axis=1).replace(0, np.nan), axis=0) * 100
        print("Sentiment label coverage by", grouping)
        print(feedback.groupby(grouping)["perplexity_sentiment"].agg(["count", "size"]))
        if not pct.empty:
            pct.plot.bar(title=f"Perplexity sentiment by {grouping}", ylabel="Labeled feedback (%)")
            plt.tight_layout()
            plt.show()

## Power Users vs\. Normal Users \- Plots:

### AI Shortcut Usage by User Group \- Bar Plot:

In [ ]:
df = load_kogod()
shortcut_df = df.loc[df["user_group"].isin(GROUP_ORDER) & df["ai_shortcut"].isin(["Yes", "No"])]
shortcut_group = pd.crosstab(shortcut_df["user_group"], shortcut_df["ai_shortcut"]).reindex(index=GROUP_ORDER, columns=["No", "Yes"], fill_value=0)
shortcut_pct = shortcut_group.div(shortcut_group.sum(axis=1).replace(0, np.nan), axis=0) * 100
shortcut_pct["Yes"].plot.bar(title="Reported AI shortcut use by user group", ylabel="Valid Yes/No responses (%)")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

## Appendix plots:

### Figure 1 \- Plot:

In [ ]:
df = load_kogod()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = load_kogod()

usage_order = ["0", "1-3", "4-7", "8-10", "11+"]

counts = df["usage_bucket"].value_counts().reindex(usage_order)
percent = df["usage_bucket"].value_counts(normalize=True).reindex(usage_order)

plt.figure(figsize=(6,5))

bars = plt.bar(
    counts.index,
    counts.values,
    color="#5B6CFF",
    edgecolor="#5B6CFF"
)

plt.title("AI Usage Frequency per Week", fontsize=12)
plt.xlabel("Times per week using AI")
plt.ylabel("Number of students")

for spine in ["top", "right"]:
    plt.gca().spines[spine].set_visible(False)

plt.grid(axis="y", linestyle="-", linewidth=0.5, alpha=0.3)

for i, (c, p) in enumerate(zip(counts.values, percent.values)):
    plt.text(
        i,
        c + 3,
        f"{p*100:.1f}%",
        ha="center",
        fontsize=10
    )

plt.tight_layout()
plt.show()

### Figure 2 \- Plot:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = load_kogod()

counts = df["model_favor"].value_counts()
percent = df["model_favor"].value_counts(normalize=True)

plt.figure(figsize=(7,5))

bars = plt.bar(
    counts.index,
    counts.values,
    color="#5B6CFF",
    edgecolor="#5B6CFF"
)

plt.title("Favorite AI Tool Among Kogod Students", fontsize=12)
plt.xlabel("Favorite AI tool")
plt.ylabel("Number of students")

# rotate x labels
plt.xticks(rotation=35, ha="right")

# clean look
for spine in ["top", "right"]:
    plt.gca().spines[spine].set_visible(False)

plt.grid(axis="y", linestyle="-", linewidth=0.5, alpha=0.3)

# add % labels
for i, (c, p) in enumerate(zip(counts.values, percent.values)):
    plt.text(
        i,
        c + 1,
        f"{p*100:.1f}%",
        ha="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()

### Perplexity usage

Only explicit Yes/No answers enter this chart. Missing responses are reported separately and are not counted as No. Responses pooled across years may reflect differences in survey availability.

In [ ]:
# Perplexity usage chart follows; missing responses are reported separately.

In [ ]:
df = load_kogod()
valid_usage = df.loc[df["perplexity_usage"].isin(["Yes", "No"]), "perplexity_usage"]
print("Missing/unrecognized Perplexity usage answers:", len(df) - len(valid_usage))
counts = valid_usage.value_counts().reindex(["No", "Yes"], fill_value=0)
counts.plot.bar(title="Reported Perplexity usage among Kogod respondents", ylabel="Valid responses")
plt.tight_layout()
plt.show()

In [ ]:
df = load_kogod()
factor = df["model_factor"].dropna()
counts = factor.value_counts()
ax = counts.plot.barh(figsize=(9, 6), title="Reported most important factor in AI tool choice")
ax.set_xlabel("Responses")
plt.tight_layout()
plt.show()

### Figure 5 \- Plot:

In [ ]:
shortcut_pct.reindex(GROUP_ORDER).plot.bar(title="Reported AI shortcut use by user group", ylabel="Valid Yes/No responses (%)")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

## Interpretation and limitations

These are descriptive survey comparisons, subject to selection, item nonresponse, and possible instrument changes across years. No causal effects, population representativeness, or within-person trajectories are established. LLM labels are supplied inputs with undocumented validation. See README.md and REVISION_NOTES.md for provenance and verification.